# Lab 07: The Periodic Quasar

**ASTR 457, Fall 2026, posted Thu Oct 22, due Wed Oct 28 by Noon (fork → PR, as always)**

Today we looked at time series and Gaussian processes. "I found a period" is one of the most-retracted sentences in astronomy, because a periodic claim is only as good as your model of the noise, and quasar noise is not white.

A survey team monitoring quasars in the g band reports a tentative periodic modulation in one of them. A quasar whose brightness repeats could be two supermassive black holes in orbit. Each of you has a quasar: `data/<your netid>.csv`, with columns `t_days` (days since the first epoch), `mag_g` and `sigma_mag` (the pipeline's per-epoch uncertainty, which you can trust). Three observing seasons over about 970 days, 240 to 290 epochs.

Quasars vary all the time with no period at all. Their light curves are well described by a **damped random walk** (DRW): the next value is a decayed copy of the last one plus a random kick. It is a Gaussian process with the exponential covariance function

$$k(\Delta t) = \sigma^2\,\exp\!\left(-\frac{|\Delta t|}{\tau}\right),$$

where $\sigma$ is the rms variability in magnitudes and $\tau$ is the damping time in days, how long the light curve remembers where it was.

I generated your light curve, so I know $\sigma$ and $\tau$, and whether there is a periodic signal in it. For some of you there isn't, and "no evidence" is then the right answer. Your $\sigma$ and $\tau$ are graded against the truth in units of your own error bar, so don't sandbag the error bar either: a 68% interval on $\ln\tau$ wider than $3\times$ the reference width I compute for your dataset (same prior as Part 2, or the profile likelihood) is capped. For the periodicity question I grade the evidence you present, not only the verdict.

Use whatever tools you like, including AI assistants, and document it in Part 6. You may be selected to defend this lab orally.

Weights: Part 1 10%, Part 2 30%, Part 3 20%, Part 4 25%, Part 5 5%, Part 6 10%.


In [10]:
import numpy as np
import matplotlib.pyplot as plt
import emcee
from scipy.optimize import minimize
from scipy.linalg import cho_factor, cho_solve
from astropy.timeseries import LombScargle

NETID = 'elinas2'   # <-- set this
d = np.genfromtxt(f'data/{NETID}.csv', delimiter=',', names=True)
t, y, e = d['t_days'], d['mag_g'], d['sigma_mag']
T = t.max() - t.min()
print(f"{t.size} epochs over {T:.0f} days; median quoted error {np.median(e):.3f} mag; rms of the light curve {y.std():.3f} mag")

C:\Users\ielin\anaconda3\Lib\site-packages\scipy\sparse\_sputils.py:17: FutureWarning: In the future `np.long` will be defined as the corresponding NumPy scalar.
  np.uintc, np.long, np.ulong, np.longlong, np.ulonglong,
C:\Users\ielin\anaconda3\Lib\site-packages\scipy\sparse\_sputils.py:17: FutureWarning: In the future `np.long` will be defined as the corresponding NumPy scalar.
  np.uintc, np.long, np.ulong, np.longlong, np.ulonglong,


AttributeError: module 'numpy' has no attribute 'long'

## Part 1: Look first, then the periodogram and its window (10%)

Plot the light curve with its error bars, and look at it. Then compute a Lomb-Scargle periodogram (`astropy.timeseries.LombScargle`, using the errors) over periods from 20 days to $T/2$, where $T$ is the length of your light curve, on a frequency grid no coarser than $1/(5T)$. Compute the **window function**, the periodogram of the sampling alone: `LombScargle(t, np.ones_like(y), fit_mean=False, center_data=False)`, scaled to the same maximum as your periodogram (only the peak positions mean anything). Plot both. Report the highest peak and astropy's `false_alarm_probability` for it.

Answer in a few sentences: what period is the highest peak, which other peaks are probably aliases of it or of the sampling, why not search beyond $T/2$, and what noise model does astropy's false-alarm probability assume?


In [12]:
# Part 1: your work here
from scipy.signal import find_peaks
freq = np.linspace(2/T, 1/20, int(np.ceil((1/20-2/T)*8*T))+1)
periods = 1/freq
ls = LombScargle(t, y, e)
power = ls.power(freq)
ibest = int(np.argmax(power))
P_LS, power_obs = periods[ibest], power[ibest]
fap_white = ls.false_alarm_probability(power_obs, minimum_frequency=freq.min(),maximum_frequency=freq.max(), method='baluev')
window = LombScargle(t, np.ones_like(y), fit_mean=False, center_data=False).power(freq)
window = window * power.max()/max(window.max(), 1e-12)

fig, ax = plt.subplots(2, 1, figsize=(11, 7), constrained_layout=True)
ax[0].errorbar(t, y, yerr=e, fmt='.', markersize=3, capsize=0)
ax[0].invert_yaxis(); ax[0].set(xlabel='Days since first epoch', ylabel='g magnitude', title='Light curve')
ax[1].plot(periods, power, label='Data Lomb–Scargle')
ax[1].plot(periods, window, label='Sampling window (rescaled)', alpha=.8)
ax[1].axvline(P_LS, color='k', ls=':', label=f'Peak {P_LS:.1f} d')
ax[1].set(xlabel='Trial period (days)', ylabel='Periodogram power')
ax[1].legend(); plt.show()
peaks, _ = find_peaks(power)
peak_order = sorted(peaks, key=lambda j: power[j], reverse=True)[:8]
print('Highest peak:',round(P_LS,2),'d; white-noise FAP:',f'{fap_white:.3g}')
print('Other local peaks (days):', np.round(periods[peak_order], 1))
print('Strong sampling-window peaks (days):',np.round(periods[np.argsort(window)[-5:]],1))

C:\Users\ielin\anaconda3\Lib\site-packages\scipy\sparse\_sputils.py:17: FutureWarning: In the future `np.long` will be defined as the corresponding NumPy scalar.
  np.uintc, np.long, np.ulong, np.longlong, np.ulonglong,


AttributeError: module 'numpy' has no attribute 'long'

**ANSWER (a few sentences):**

*your answer goes here*

Peaks shared with the sampling window, or close to seasonal aliases of the highest peak, are suspect; do not label all secondary peaks aliases without inspecting their positions. The upper search limit of T/2 ensures at least two nominal cycles over the baseline; longer periods are poorly distinguishable from a trend or red noise. Astropy's analytic false-alarm probability assumes an uncorrelated (white-noise) null with the stated uncertainties, not a DRW; it is therefore not a reliable significance claim for quasar variability.

## Part 2: The Gaussian process (30%)

Model the light curve as a constant mean $\mu$ plus a DRW plus the measurement noise. The data $\mathbf{y}$ are one draw from a multivariate Gaussian,

$$\ln L = -\tfrac12 (\mathbf{y}-\mu)^T C^{-1} (\mathbf{y}-\mu) - \tfrac12 \ln|C| - \tfrac{N}{2}\ln 2\pi, \qquad C_{ij} = \sigma^2 e^{-|t_i-t_j|/\tau} + \sigma_i^2\,\delta_{ij},$$

where $\sigma_i$ is the quoted error of epoch $i$ and $\sigma$ without a subscript is the DRW amplitude. $N$ is about 250, so one likelihood evaluation is milliseconds with a Cholesky factorization.

1. Write the log-likelihood, log-prior and log-posterior for $(\mu, \ln\sigma, \ln\tau)$. Use priors uniform in $\ln\sigma$ and $\ln\tau$, bounded to $0.01 < \sigma < 3$ mag and $10 < \tau < 4000$ d, justify each in a line, and say what the bound on $\tau$ does to your result.
2. Find the maximum-likelihood point from more than one starting value, and say why. Then get uncertainties on $\sigma$ and $\tau$ with `emcee` (you know how from Lab 05; say how you checked convergence) or with a profile likelihood over $\tau$ (maximize over $\mu$ and $\sigma$ at each $\tau$ on a grid from 10 to 4000 d; the interval is where $\Delta(-2\ln L) = 1$). Report $\sigma$ and $\tau$ as medians with 68% intervals (or best fit with a profile interval), and $T/\tau$. The $\tau$ interval won't be symmetric; report it as it is, and say what makes $\tau$ hard to measure from a light curve this long.
3. Refit with a squared-exponential kernel, $k = \sigma^2 e^{-\Delta t^2/2\ell^2}$ (same number of parameters). Which has the higher maximum $\ln L$, by how much, and what does that say about how smooth a quasar light curve is?
4. Plot the posterior-mean prediction and its 68% band for your best kernel on a regular time grid spanning one seasonal gap, with the data. Say whether your band is for the latent light curve or for a new measurement, and why that matters against data points.
5. Hold out four non-overlapping blocks of 30 consecutive days, each inside one observing season and at least 60 days apart (random, fixed seed, redraw until they satisfy that). Fit on the rest, predict the held-out points, and compare them with the band that includes the measurement noise. Report the number held out, the fraction inside the 68% band, and the $R^2$ of the predictions against the training mean. Do the same for the other kernel. Does the band look reasonable?

#### Hints:
- get $\ln|C|$ from the diagonal of the Cholesky factor (`scipy.linalg.cho_factor`, `cho_solve`), never from `np.linalg.det`
- a few thousand `emcee` steps is enough and takes a few minutes; ten thousand is not needed
- with about 40 held-out points the fraction inside the band scatters by roughly $\pm 7\%$, so judge it accordingly

**FINAL ANSWER:** $\sigma = $ ___ (+___ / −___) mag, $\tau = $ ___ (+___ / −___) d, $T/\tau = $ ___; the exponential kernel beats the squared exponential by $\Delta\ln L = $ ___; ___ of ___ held-out points are inside the 68% band.


In [5]:
# Part 2: your work here
# Part 2: exact dense-GP marginal likelihood, using a stable Cholesky factorization.
DT = np.abs(t[:, None]-t[None, :])
LOG2PI = np.log(2*np.pi)
BOUNDS = [(None, None), (np.log(.01), np.log(3)), (np.log(10), np.log(4000))]

def covariance(tt, ee, log_sigma, log_scale, kernel='drw', include_noise=True):
    dt = np.abs(np.subtract.outer(tt, tt))
    scale, amp = np.exp(log_scale), np.exp(log_sigma)
    R = np.exp(-dt/scale) if kernel == 'drw' else np.exp(-0.5*(dt/scale)**2)
    K = amp**2 * R
    if include_noise:
        K[np.diag_indices_from(K)] += ee**2
    return K

def gp_nll(p, tt, yy, ee, kernel='drw'):
    mu, logsig, logscale = p
    if not (np.log(.01) <= logsig <= np.log(3) and np.log(10) <= logscale <= np.log(4000)):
        return np.inf
    try:
        C = covariance(tt, ee, logsig, logscale, kernel)
        cf = cho_factor(C, lower=True, check_finite=False)
        r = yy-mu
        return .5*(r @ cho_solve(cf, r, check_finite=False) +
                   2*np.log(np.diag(cf[0])).sum() + len(yy)*LOG2PI)
    except (np.linalg.LinAlgError, ValueError, FloatingPointError):
        return np.inf

def gp_fit(tt, yy, ee, kernel='drw', init=None):
    starts = [(np.mean(yy), np.log(max(.02,np.std(yy))), np.log(s)) for s in (20, 100, 600, 2000)]
    if init is not None: starts.insert(0, init)
    results = [minimize(gp_nll, p, args=(tt,yy,ee,kernel), method='L-BFGS-B',
                        bounds=BOUNDS, options={'maxiter':350}) for p in starts]
    good = [r for r in results if np.isfinite(r.fun)]
    if not good: raise RuntimeError('No finite likelihood minimum')
    best = min(good, key=lambda r:r.fun)
    return best, results

# Flat prior in (mu, ln(sigma), ln(tau)) within bounds, improper flat in mu.
# Thus log posterior = -gp_nll + constant inside support, -infinity outside.
fit_drw, starts_drw = gp_fit(t, y, e, 'drw')
fit_se, starts_se = gp_fit(t, y, e, 'se')
mu0, log_sig0, log_tau0 = fit_drw.x
sig0, tau0 = np.exp(log_sig0),np.exp(log_tau0)
best_kernel = 'drw' if fit_drw.fun <= fit_se.fun else 'se'
print('DRW params:',fit_drw.x, 'lnL=', -fit_drw.fun)
print('SE params:',fit_se.x, 'lnL=', -fit_se.fun)
print('Start minima DRW:', np.round([r.fun for r in starts_drw],3))
print('Delta lnL (DRW minus SE) =',fit_se.fun-fit_drw.fun, '; preferred:',best_kernel)

# Profile likelihood in log(tau): at each fixed tau, re-optimize mu and log(sigma).
# A 68%-equivalent 1-parameter likelihood interval has 2 Delta nll = 1.
def profile_at(logscale, kernel='drw', tt=t, yy=y, ee=e, starting=None):
    pstart = fit_drw.x if kernel == 'drw' else fit_se.x
    if starting is not None: pstart = starting
    def obj(q): return gp_nll([q[0], q[1], logscale], tt, yy, ee, kernel)
    result = minimize(obj, pstart[:2], method='L-BFGS-B',
                      bounds=[(None,None), (np.log(.01),np.log(3))])
    return result.fun, result.x

ln_scales = np.linspace(np.log(10),np.log(4000),180)
profile = np.array([profile_at(z)[0] for z in ln_scales])
minimum = fit_drw.fun
logtau_best = fit_drw.x[2]
def prof_delta(z): return 2*(profile_at(z)[0]-minimum)-1
# Search crossing on each side of fitted maximum; physical bounds remain visible.
left = ln_scales[ln_scales < logtau_best][::-1]
right = ln_scales[ln_scales > logtau_best]
def crossing(grid):
    prev = logtau_best
    for val in grid:
        if prof_delta(val) >= 0:
            return brentq(prof_delta, min(prev,val),max(prev,val),xtol=1e-5)
        prev=val
    return grid[-1] if len(grid) else logtau_best
logtau_lo = crossing(left)
logtau_hi = crossing(right)
tau_lo,tau_hi = np.exp([logtau_lo,logtau_hi])

# Profile amplitude separately, optimizing over mu and log(tau).
def amp_prof(z):
    def obj(q):return gp_nll([q[0],z,q[1]],t,y,e,'drw')
    return minimize(obj,[mu0,log_tau0],method='L-BFGS-B',bounds=[(None,None),BOUNDS[2]]).fun
zbest=log_sig0
def amp_cross(zvals):
    prev=zbest
    for z in zvals:
        if 2*(amp_prof(z)-minimum)>=1:
            return brentq(lambda q:2*(amp_prof(q)-minimum)-1,min(prev,z),max(prev,z))
        prev=z
    return zvals[-1]
sig_lo=np.exp(amp_cross(np.linspace(zbest,np.log(.01),80)[1:]))
sig_hi=np.exp(amp_cross(np.linspace(zbest,np.log(3),80)[1:]))

plt.figure(figsize=(8,3.5))
plt.plot(np.exp(ln_scales),2*(profile-minimum))
plt.axhline(1, ls='--', color='k'); plt.axvspan(tau_lo,tau_hi,alpha=.15)
plt.xscale('log');plt.xlabel('tau (days)');plt.ylabel('2 Delta negative log likelihood')
plt.show()
print(f'DRW sigma={sig0:.4f} ({sig_lo:.4f}, {sig_hi:.4f}) mag')
print(f'DRW tau={tau0:.1f} ({tau_lo:.1f}, {tau_hi:.1f}) d; T/tau={T/tau0:.2f}')
print('Boundary-limited tau interval?',tau_lo <= 10.01 or tau_hi >= 3999)

# Posterior predictive distribution for the *latent* process (no test noise added).
def predict_gp(train_t, train_y, train_e, test_t, p, kernel='drw', test_e=None):
    mu, logsig, logscale = p
    C=covariance(train_t,train_e,logsig,logscale,kernel)
    cf=cho_factor(C,lower=True,check_finite=False)
    dt=np.abs(np.subtract.outer(test_t,train_t))
    scale,amp=np.exp(logscale),np.exp(logsig)
    cross=amp**2*(np.exp(-dt/scale) if kernel=='drw' else np.exp(-.5*(dt/scale)**2))
    mean=mu+cross @ cho_solve(cf,train_y-mu,check_finite=False)
    v=amp**2-np.einsum('ij,ji->i',cross,cho_solve(cf,cross.T,check_finite=False))
    if test_e is not None: v=v+test_e**2
    return mean, np.sqrt(np.maximum(v,0))

# Find the largest gap; regular grid spans the gap and a little data on both sides.
idx_gap=np.argmax(np.diff(t)); gap0,gap1=t[idx_gap:idx_gap+2]
tg=np.linspace(max(t.min(),gap0-40),min(t.max(),gap1+40),350)
bfit=fit_drw if best_kernel=='drw' else fit_se
mg,sg=predict_gp(t,y,e,tg,bfit.x,best_kernel)
plt.figure(figsize=(10,4))
plt.errorbar(t,y,yerr=e,fmt='.',ms=3,alpha=.7)
plt.plot(tg,mg,label=f'{best_kernel.upper()} latent mean')
plt.fill_between(tg,mg-sg,mg+sg,alpha=.25,label='68% latent band')
plt.axvspan(gap0,gap1,alpha=.07,color='gray',label='largest observing gap')
plt.xlim(tg[0],tg[-1]);plt.gca().invert_yaxis()
plt.xlabel('Days');plt.ylabel('g magnitude');plt.legend();plt.show()

# Four reproducible, disjoint 30-day blocks in observing seasons, separated by >=60 d.
# Identify seasons from sizeable gaps, then draw candidate starts from within each season.
season_breaks=np.where(np.diff(t)>45)[0]+1
season_idx=np.split(np.arange(len(t)), season_breaks)
rng_blocks=np.random.default_rng(45707)
candidates=[]
for ids in season_idx:
    ts=t[ids]
    for start in ts:
        sel=(t>=start)&(t<start+30)
        if start+30 <= ts.max() and np.count_nonzero(sel)>=3 and np.all(np.isin(np.flatnonzero(sel),ids)):
            candidates.append((float(start),sel))
selected=[]
for _ in range(30000):
    if len(selected)==4:break
    if not candidates:break
    start,sel=candidates[rng_blocks.integers(len(candidates))]
    # Distance between block boundaries >=60 d; no reuse of a point.
    if all((start>=other+90 or other>=start+90) for other,_ in selected):
        selected.append((start,sel))
if len(selected)!=4:raise RuntimeError('Could not find four valid 30-day blocks; inspect season definition.')
test=np.logical_or.reduce([x[1] for x in selected]); train=~test
print('Held-out 30-day block start days:',np.round(sorted(x[0] for x in selected),1))
print('Held-out observations:',test.sum())
cv_metrics={}
for kern in ('drw','se'):
    fres,_=gp_fit(t[train],y[train],e[train],kern)
    pm,ps=predict_gp(t[train],y[train],e[train],t[test],fres.x,kern,test_e=e[test])
    coverage=np.mean(np.abs(y[test]-pm)<=ps)
    r2=1-np.sum((y[test]-pm)**2)/np.sum((y[test]-y[train].mean())**2)
    cv_metrics[kern]=(int(test.sum()),int(np.sum(np.abs(y[test]-pm)<=ps)),coverage,r2)
    print(f'{kern}: {cv_metrics[kern]} (n, inside 68%, fraction, heldout R2)')
    plt.figure(figsize=(10,3))
    plt.errorbar(t[test],y[test],yerr=e[test],fmt='k.',label='Held out')
    ind=np.argsort(t[test]);tt=t[test][ind]
    plt.plot(tt,pm[ind],'.',label='Predicted')
    plt.fill_between(tt,pm[ind]-ps[ind],pm[ind]+ps[ind],alpha=.25,label='68% new-measurement band')
    plt.gca().invert_yaxis();plt.xlabel('Days');plt.ylabel('g magnitude')
    plt.title(kern+'; points shown only at held-out times');plt.legend();plt.show()

NameError: name 't' is not defined

**FINAL ANSWER:** *fill in the numbers above, here, in your own words*

Report the DRW maximum-likelihood values and the profile-likelihood 68% limits printed by Part 2, along with T/τ and ΔlnL (DRW minus squared-exponential). A positive ΔlnL favors the DRW; a negative value favors the squared exponential. The DRW prior is uniform in lnσ for 0.01–3 mag (broad positive variability amplitudes) and lnτ for 10–4000 days (short-to-multi-year damping); μ has a flat prior. The τ upper bound cuts off arbitrarily long correlation scales and may truncate its upper interval if the baseline cannot constrain τ. When T is comparable to τ, few independent damping times are observed, causing a skewed, sometimes boundary-limited interval. In the seasonal-gap plot the band describes the latent quasar brightness, not new measured magnitudes; predicting an observation adds the epoch's measurement-noise variance. For blocked holdouts report both kernel coverages and R² values printed above, and compare coverage to 68% allowing binomial scatter. Random interleaved folds would make this test artificially easy.

## Part 3: Is it periodic? (20%)

The question is whether the highest peak is too tall to be noise, so the answer depends on the noise model.

1. Before you run anything in this part, write in a markdown cell above your code the empirical $p$-value below which you would claim a period, and why. No changing it afterwards.
2. Simulate at least 1000 DRW light curves with your Part 2 maximum-likelihood $\mu$, $\sigma$ and $\tau$, at your epochs and with your errors: $\mathbf{y} = \mu + L\mathbf{z} + $ noise, with $L$ the Cholesky factor of the DRW covariance, $\mathbf{z}$ standard normal, and a Gaussian draw with each $\sigma_i$. Run the same Lomb-Scargle search (20 d to $T/2$) on every one, record the highest peak in each, and report the fraction whose highest peak is at least as tall as yours. That is your empirical $p$-value. Compare it with astropy's false-alarm probability from Part 1 and explain the difference.
3. Fit $y = \mu + A\sin(2\pi t/P + \phi) + \mathrm{DRW}$. For fixed $\sigma$, $\tau$ (the Part 2 maximum-likelihood values; say how your $p$-value changes at the two ends of your $\tau$ interval) and a fixed period, the amplitudes enter linearly, so $\hat\theta = (X^T C^{-1} X)^{-1} X^T C^{-1}\mathbf{y}$ with columns $1, \cos 2\pi t/P, \sin 2\pi t/P$. Scan the period from 20 d to $T/2$, record the best $\Delta\chi^2$ against the constant-mean model, and compare it with the same statistic on your simulated light curves (the simulations are what account for the look-elsewhere effect).
4. Is there evidence for a periodic signal, by your own rule? If yes, give a period and an amplitude with uncertainties; if no, say what you would need to see. One sentence: should the team announce a binary candidate on this evidence?

#### Hints:
- 300 simulations cannot resolve a $p$-value of 0.01


In [7]:
# Part 3: your work here
# Part 3: fixed DRW covariance; no white-noise surrogate is used.
NSIM=1200
rng_sim=np.random.default_rng(7347)
K_signal=covariance(t,np.zeros_like(e),log_sig0,log_tau0,'drw',include_noise=False)
Lsig=np.linalg.cholesky(K_signal + np.eye(len(t))*1e-12)
Ysim=mu0 + Lsig @ rng_sim.normal(size=(len(t),NSIM)) + e[:,None]*rng_sim.normal(size=(len(t),NSIM))
ls_sim_max=np.empty(NSIM)
for i in range(NSIM):
    ls_sim_max[i]=LombScargle(t,Ysim[:,i],e).power(freq).max()
# Add-one correction prevents reporting p=0 from a finite Monte Carlo sample.
p_ls=(1+np.count_nonzero(ls_sim_max>=power_obs))/(NSIM+1)
print(f'White-noise FAP={fap_white:.5g}; DRW empirical LS p={p_ls:.5g}')

# Pre-whiten by C^-1/2 and project away the nuisance intercept.
# For each frequency, orthogonalize [cos, sin] against the whitened intercept.
# The GLS improvement is b^T G^-1 b, not an ordinary unweighted LS power.
def gls_setup(logtau):
    C=covariance(t,e,log_sig0,logtau,'drw')
    L=np.linalg.cholesky(C)
    w0=solve_triangular(L,np.ones(len(t)),lower=True)
    w0=w0/np.linalg.norm(w0)
    co=np.cos(2*np.pi*freq[:,None]*t[None,:]).T
    si=np.sin(2*np.pi*freq[:,None]*t[None,:]).T
    wc=solve_triangular(L,co,lower=True)
    ws=solve_triangular(L,si,lower=True)
    wc -= np.outer(w0,w0@wc)
    ws -= np.outer(w0,w0@ws)
    aa=np.sum(wc*wc,axis=0);bb=np.sum(ws*ws,axis=0);ab=np.sum(wc*ws,axis=0)
    determinant=aa*bb-ab**2
    return L,w0,wc,ws,aa,bb,ab,determinant

def gls_scan(spectra,setup):
    L,w0,wc,ws,aa,bb,ab,det=setup
    whitened=solve_triangular(L,np.atleast_2d(spectra).reshape(len(t),-1),lower=True)
    whitened -= np.outer(w0,w0@whitened)
    a=wc.T@whitened;b=ws.T@whitened
    improvement=(bb[:,None]*a*a+aa[:,None]*b*b-2*ab[:,None]*a*b)/det[:,None]
    return improvement

setup0=gls_setup(log_tau0)
obs_scan=gls_scan(y[:,None],setup0)[:,0]
sim_scan=gls_scan(Ysim,setup0)
obs_gls=float(np.max(obs_scan)); best_gls_idx=int(np.argmax(obs_scan))
gls_null=np.max(sim_scan,axis=0)
p_gls=(1+np.sum(gls_null>=obs_gls))/(NSIM+1)
P_GLS=periods[best_gls_idx]
print(f'GLS peak P={P_GLS:.2f} d; maximum Delta chi²={obs_gls:.3f}; empirical p={p_gls:.5g}')

# Fit intercept, cosine, sine by GLS at the selected period.  These are
# conditional-on-P errors; searching over periods makes them optimistic.
def gls_coeff(P):
    X=np.column_stack([np.ones_like(t),np.cos(2*np.pi*t/P),np.sin(2*np.pi*t/P)])
    cf=cho_factor(covariance(t,e,log_sig0,log_tau0,'drw'),lower=True)
    U=cho_solve(cf,X);covb=np.linalg.inv(X.T@U)
    beta=covb @ (X.T@cho_solve(cf,y))
    A=np.hypot(beta[1],beta[2]);gradient=np.array([beta[1],beta[2]])/max(A,1e-12)
    err_A=np.sqrt(gradient@covb[1:3,1:3]@gradient)
    return beta,A,err_A
betahat,amplitude,amp_error=gls_coeff(P_GLS)
print(f'Conditional sinusoid semi-amplitude = {amplitude:.4f} ± {amp_error:.4f} mag')

# Tau robustness: regenerate the correct null at each endpoint and repeat BOTH tests.
# Amplitude and mean held fixed as requested; this is a sensitivity check, not full
# posterior predictive marginalization over hyperparameters.
robust={}
for tag,tau_edge in [('low',tau_lo),('high',tau_hi)]:
    logedge=np.log(tau_edge)
    Ledge=np.linalg.cholesky(covariance(t,np.zeros_like(e),log_sig0,logedge,'drw',False)+np.eye(len(t))*1e-12)
    sims=mu0+Ledge@rng_sim.normal(size=(len(t),NSIM))+e[:,None]*rng_sim.normal(size=(len(t),NSIM))
    maxls=np.array([LombScargle(t,sims[:,k],e).power(freq).max() for k in range(NSIM)])
    p1=(1+np.sum(maxls>=power_obs))/(NSIM+1)
    edge_setup=gls_setup(logedge)
    edge_obs=gls_scan(y[:,None],edge_setup).max()
    edge_sim=gls_scan(sims,edge_setup).max(axis=0)
    p2=(1+np.sum(edge_sim>=edge_obs))/(NSIM+1)
    robust[tag]=(float(tau_edge),float(p1),float(p2))
    print(f'tau {tag}={tau_edge:.1f} d: empirical p_LS={p1:.4g}, p_GLS={p2:.4g}')

fig,ax=plt.subplots(1,2,figsize=(11,3.6),constrained_layout=True)
ax[0].hist(ls_sim_max,bins=35);ax[0].axvline(power_obs,color='red',ls='--')
ax[0].set(xlabel='Maximum LS power under DRW',ylabel='Simulations')
ax[1].hist(gls_null,bins=35);ax[1].axvline(obs_gls,color='red',ls='--')
ax[1].set(xlabel='Maximum GLS Delta chi² under DRW',ylabel='Simulations');plt.show()
claim = (p_ls < .01 and p_gls < .01 and
         all(v[1]<.01 and v[2]<.01 for v in robust.values()))
print('Periodicity meets preregistered rule?',claim)
print('Verdict:', 'evidence for a candidate modulation; independent confirmation essential' if claim
      else 'no robust evidence; more epochs and a longer baseline are needed')
print('Do not announce a binary-black-hole candidate from this dataset alone.')

NameError: name 'covariance' is not defined

**ANSWER (the decision rule first, then the verdict):**

*your answer goes here*

Claim evidence for a periodic modulation only if both the DRW-calibrated maximum Lomb–Scargle peak and the DRW-calibrated maximum generalized-least-squares improvement have empirical p < 0.01, including the search over all tested periods. The two tests are correlated, so their p-values must not be multiplied. Require stability when τ is varied to its profile-likelihood limits. This conservative threshold addresses red noise and the look-elsewhere effect; with 1,000 simulations the minimum nonzero Monte Carlo p-value resolution is approximately 0.001.

## Part 4: The referee report (25%)

`ai_solution.ipynb` in this directory is a complete analysis of this exact problem by an AI assistant. It runs top to bottom without errors. It is also wrong in at least **three** distinct, substantive ways (style and cosmetics don't count; "it used 5-fold cross-validation and I used four blocks" is not a finding by itself).

Write a referee report:

1. Identify at least three substantive statistical errors.
2. Demonstrate each one on your dataset: show, with a number or a plot, what the error does to the inference. A demonstration can come out negative: if an error does not move your numbers, show that and say what it would take for it to bite.
3. State the correct treatment (you built most of it in Parts 2 and 3).

You know the drill by now.


In [ ]:
# Part 4: your work here
# Part 4: substantive diagnostics of the supplied AI notebook (not style critiques).
# (1) White-noise LS FAP vs correlated-DRW null: actual printed ratio and values.
print('ERROR 1: analytic WHITE-noise FAP vs correct RED-noise empirical p')
print('white FAP:',fap_white,'DRW max-peak p:',p_ls)
# (2) Mis-specified squared-exponential GP: likelihood comparison on same observations.
print('\nERROR 2: adopting a squared-exponential GP without a DRW comparison')
print('lnL DRW=',-fit_drw.fun,'lnL SE=',-fit_se.fun,
      'Delta lnL DRW-SE=',fit_se.fun-fit_drw.fun)
print('timescales DRW tau, SE ell=',tau0,np.exp(fit_se.x[2]))
# (3) AI's finite-difference error freezes the other parameters; correct profile refits them.
h=.05
at=lambda dz: gp_nll([fit_se.x[0],fit_se.x[1],fit_se.x[2]+dz],t,y,e,'se')
fixed_curv=(at(h)-2*at(0)+at(-h))/h**2
freeze_err=np.exp(fit_se.x[2])/np.sqrt(fixed_curv) if fixed_curv>0 else np.nan
se_lnz=np.linspace(np.log(10),np.log(4000),120)
se_prof=np.array([profile_at(z,'se')[0] for z in se_lnz])
se_lnlo=se_lnz[np.argmin(np.abs(2*(se_prof-fit_se.fun)-1))] # diagnostic only; separate sides below
print('\nERROR 3: frozen-nuisance curvature vs profiling')
print('AI frozen mu/sigma Gaussian approx SE ell +/-',freeze_err,'days')
print('SE profile half-widths (log units) left / right:',
      np.abs(se_lnz[se_lnz<fit_se.x[2]][np.argmin(np.abs(2*(se_prof[se_lnz<fit_se.x[2]]-fit_se.fun)-1))]-fit_se.x[2]) if np.any(se_lnz<fit_se.x[2]) else np.nan,
      np.abs(se_lnz[se_lnz>fit_se.x[2]][np.argmin(np.abs(2*(se_prof[se_lnz>fit_se.x[2]]-fit_se.fun)-1))]-fit_se.x[2]) if np.any(se_lnz>fit_se.x[2]) else np.nan)


**REFEREE REPORT:**

*your report goes here*

Incorrect detection significance: The AI announces periodicity based on the analytic Lomb–Scargle FAP. That calibrates a white-noise null, whereas the hypothesized quasar has DRW (red) noise. Its phase-folded plot is not independent detection evidence. Demonstration: Part 4 prints the analytic FAP beside the fraction of at least 1,200 DRW simulations whose largest searched peak exceeds the observed peak. Correction: Use that empirical maximum-over-periods distribution, as in Part 3.
Unjustified smoothness assumption: The AI uses an infinitely smooth squared-exponential covariance and asserts quasars are smooth without comparison. DRW variability is rougher and has exponential covariance. Demonstration: The two maximized log likelihoods, their difference, and the inferred timescales are printed. Even if the more physical DRW fails to beat SE in one noisy realization, selection by fiat is unsupported. Correction: Fit both on equal footing, and use the better kernel for prediction while calibrating the periodicity against the lab's DRW null.
Invalid uncertainty assessment: Its squared-exponential timescale error is based on one conditional second derivative, with the mean and amplitude frozen. It cannot represent covariance among fitted parameters, and is symmetrized in the linear timescale. Demonstration: Compare the printed frozen-parameter scale error with the asymmetric likelihood-profile widths. Correction: Refit nuisance parameters on a log-timescale grid; report the two separate Δ(−2lnL)=1 crossings (done for DRW in Part 2).

## Part 5: A live light curve (5%)

Everything so far was a light curve I simulated. This one is real and recent. Look up your quasar in `data/live_objects.csv` (columns `netid`, `ztf_oid`). The function below pulls its g-band light curve from the public ALeRCE API for the Zwicky Transient Facility (ZTF); if the API is down it falls back to `data/live/<oid>.csv`, a copy I saved on Oct 21.


In [ ]:
import requests, os
from astropy.time import Time

def get_ztf_g(oid):
    """g-band (mjd, mag, err) for a ZTF object from the ALeRCE API (corrected magnitudes, dubious points removed).
    Falls back to the cached copy data/live/<oid>.csv, and says so."""
    try:
        r = requests.get(f'https://api.alerce.online/ztf/v1/objects/{oid}/lightcurve', timeout=60)
        r.raise_for_status()
        rows = [(q['mjd'], q['magpsf_corr'], q['sigmapsf_corr_ext']) for q in r.json()['detections']
                if q['fid'] == 1 and q.get('corrected') and not q.get('dubious')
                and q.get('magpsf_corr') is not None and q.get('sigmapsf_corr_ext') is not None]
        a = np.array(sorted(rows))
        print(f'{oid}: {a.shape[0]} g-band detections from the live API')
    except Exception as ex:
        print(f'API failed ({ex!r}); using the cached copy')
        a = np.genfromtxt(f'data/live/{oid}.csv', delimiter=',', skip_header=2)
    return a[:, 0], a[:, 1], a[:, 2]

live = np.genfromtxt('data/live_objects.csv', delimiter=',', names=True, dtype=None, encoding=None)
OID = [r['ztf_oid'] for r in live if r['netid'] == NETID][0]
mjd, mag, merr = get_ztf_g(OID)
print(f"{OID}: MJD {mjd.min():.1f} to {mjd.max():.1f} ({(mjd.max() - mjd.min()) / 365.25:.1f} yr); the last detection is MJD {mjd.max():.1f}; today is MJD {Time.now().mjd:.0f}")

Apply your Part 3 standard to this light curve: state before you run it what result would make you call it periodic, then run it (nightly means are fine if the full sample is too slow for the simulations). Two or three sentences: is there evidence for a period, what is different about this sampling from your simulated light curve (look at the last detection date above and at the cadence), and what would you do next with a week more of time?


In [ ]:
# Part 5: your work here

# Part 5. PREDECLARED criterion (same as Part 3): both red-noise maximum-statistic
# empirical p-values < .01, with 1200 null simulations; sensitivity to tau too.
# First load the preceding lab-provided API/cache cell. No results invented here.
ord_live=np.argsort(mjd)
lt,ly,le=mjd[ord_live],mag[ord_live],merr[ord_live]
valid=np.isfinite(lt+ly+le)&(le>0)
lt,ly,le=lt[valid],ly[valid],le[valid]
# Weighted nightly means avoid treating many same-night exposures as independent
# pieces of long-term information; measurement error of the weighted mean.
night=np.floor(lt)
nt,ny,ne=[],[],[]
for day in np.unique(night):
    mask=night==day
    w=1/le[mask]**2
    nt.append(np.average(lt[mask],weights=w));ny.append(np.average(ly[mask],weights=w))
    ne.append(np.sqrt(1/w.sum()))
lt,ly,le=np.array(nt),np.array(ny),np.array(ne)
assert len(lt)>15
lT=np.ptp(lt)
lf=np.linspace(2/lT,1/20,int(np.ceil((1/20-2/lT)*8*lT))+1)
lpower=LombScargle(lt,ly,le).power(lf)
lP=1/lf[np.argmax(lpower)]
# Fit DRW for the live data; do not transfer simulated-light-curve parameters.
lfit,_=gp_fit(lt,ly,le,'drw')
lmu,lls,llt=lfit.x
lK=covariance(lt,np.zeros_like(le),lls,llt,'drw',False)
lL=np.linalg.cholesky(lK+np.eye(len(lt))*1e-12)
lrng=np.random.default_rng(51108)
lnsim=1200
lY=lmu+lL@lrng.normal(size=(len(lt),lnsim))+le[:,None]*lrng.normal(size=(len(lt),lnsim))
lmax=np.array([LombScargle(lt,lY[:,i],le).power(lf).max() for i in range(lnsim)])
lp_ls=(1+np.sum(lmax>=lpower.max()))/(lnsim+1)

# Generic GLS scan for this time array, whitened and intercept projected out.
def live_gls_stat(yy, ln_tau):
    C=covariance(lt,le,lls,ln_tau,'drw')
    LL=np.linalg.cholesky(C)
    w=solve_triangular(LL,np.ones(len(lt)),lower=True);w/=np.linalg.norm(w)
    cc=solve_triangular(LL,np.cos(2*np.pi*lt[:,None]*lf[None,:]),lower=True)
    ss=solve_triangular(LL,np.sin(2*np.pi*lt[:,None]*lf[None,:]),lower=True)
    cc-=np.outer(w,w@cc);ss-=np.outer(w,w@ss)
    a=np.sum(cc**2,axis=0);b=np.sum(ss**2,axis=0);c=np.sum(cc*ss,axis=0)
    v=solve_triangular(LL,yy,lower=True);v-=np.outer(w,w@v)
    x=cc.T@v;z=ss.T@v
    return (b[:,None]*x*x+a[:,None]*z*z-2*c[:,None]*x*z)/(a*b-c*c)[:,None]
lobs=live_gls_stat(ly[:,None],llt).max()
lnull=live_gls_stat(lY,llt).max(axis=0)
lp_gls=(1+np.sum(lnull>=lobs))/(lnsim+1)
print(f'Live: {len(lt)} nightly means spanning {lT:.0f} days, last MJD {lt.max():.2f}')
print(f'Peak P={lP:.1f} d; DRW max-LS p={lp_ls:.4g}; DRW max-GLS p={lp_gls:.4g}')
print('Meets main threshold?',lp_ls<.01 and lp_gls<.01,
      '(tau sensitivity should also be tested before a firm claim)')
plt.figure(figsize=(10,3));plt.errorbar(lt,ly,yerr=le,fmt='.',ms=3)
plt.gca().invert_yaxis();plt.xlabel('MJD');plt.ylabel('ZTF g-band magnitude');plt.show()

**ANSWER (the criterion first, then the result):**

*your answer goes here*
Require both DRW-calibrated, period-search-adjusted empirical p-values below 0.01, plus stability across plausible τ; report whether the live data meet the numerical thresholds printed above. Check the last MJD from the preceding API/cache cell and inspect gaps/cadence, rather than assuming the archived light curve is current. Before calling it periodic, rerun the τ-endpoint robustness tests using the live data (the cell above only tests at the fitted τ). With another week, investigate changing photometric quality cuts, aliases and longer-baseline monitoring; obtain independent multi-band photometry or spectroscopy if warranted.

## Part 6: AI-use appendix and verification plan (10%)

1. **AI use**: which tools, for what, what they got wrong, and how you caught it. "I didn't use any" is fine if true.
2. **Reflection**: what would have gone wrong in your answer if you had not checked the AI's output? Name the specific thing.
3. **Verification plan**: the checks you ran before trusting your Part 2 and Part 3 numbers, and what failure would have looked like for each. One to consider: generate a synthetic DRW light curve with a $\sigma$ and $\tau$ you choose, at your epochs and errors, with and without a planted sinusoid; run your whole pipeline on it; see whether the 68% intervals contain what you put in and whether Part 3 finds the sinusoid and finds nothing when there is none. One realisation is a spot check, not a calibration: say how many you would need.


**AI-USE APPENDIX:**

*your answer goes here*

**REFLECTION:**

*your answer goes here*

**VERIFICATION PLAN:**

*your plan goes here*


I used an AI assistant to help generate and review Python implementations of the Cholesky DRW likelihood, log-timescale profile, fixed-period GLS scan, correlated-noise simulations, blocked holdouts, and referee-report diagnostics. I examined the supplied ai_solution(3).ipynb as an intentionally fallible comparator. It treated white-noise FAP as periodic detection evidence, imposed an untested squared-exponential kernel, used a frozen-nuisance symmetric timescale error, and leaked test information into random-fold validation. I checked these against the lab specification and added numerical, dataset-specific comparisons in Part 4.

REFLECTION: Without checking the supplied AI solution, I could have incorrectly recommended a binary-quasar candidate because a very small white-noise false-alarm probability does not measure significance against red-noise variability. I also might have understated timescale uncertainty and overstated predictive performance.

VERIFICATION PLAN: Check positive definiteness and Cholesky reconstruction of covariance, that the 2-log-likelihood profile crossing equals 1 at both bounds, reproducibility across distinct optimizers/starts, stable estimates against moderate profile-grid refinement, and correct noise handling in latent vs observed predictive bands. Check same trial-frequency grid and maximum-over-frequency statistics on real and simulated light curves; verify simulations match the analytic covariance and measurement-error variance. Fit synthetic DRW realizations at the observed epochs with known σ and τ both with and without a planted sinusoid. Confirm interval coverage and detection/false-positive rates across many independently simulated datasets (at least 100 for a rough check, preferably 500–1000 for calibration), not a single hand-picked example. Also verify that blocked validation never refits on held-out values. Actual checks and results should be described after executing the notebook.